In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Quick script to summarize sequence length distribution
Input:  txt file, one sequence per line
Output: print stats (count, min, max, mean, median, quartiles) + histogram preview
"""

import sys
import numpy as np
from collections import Counter

# ==== 用户参数 ====
INPUT_TXT = "/Users/shulei/PycharmProjects/Dataset/scripts/test_set/matrix_out/sparse_matrix_v0.3.1_row_ids.txt"   # 改成你的序列文件路径
TOP_N = 99999  # 打印最常见的长度

def main():
    # 读取所有非空行
    with open(INPUT_TXT) as f:
        seqs = [line.strip() for line in f if line.strip()]
    lengths = [len(s) for s in seqs]

    if not lengths:
        print("❌ No sequences found.")
        return

    arr = np.array(lengths)
    print(f"Total sequences: {len(arr)}")
    print(f"Min length    : {arr.min()}")
    print(f"Max length    : {arr.max()}")
    print(f"Mean length   : {arr.mean():.2f}")
    print(f"Median length : {np.median(arr):.2f}")
    print(f"Q1 (25%)      : {np.percentile(arr, 25):.2f}")
    print(f"Q3 (75%)      : {np.percentile(arr, 75):.2f}")

    # 打印最常见的长度分布
    counter = Counter(lengths)
    print(f"\nTop {TOP_N} most common lengths:")
    for length, count in counter.most_common(TOP_N):
        print(f"  length {length:4d} : {count} seqs")

    # 简单直方图（文本版）
    print("\nHistogram (bin=50 aa):")
    bins = range((arr.min() // 50) * 50, arr.max() + 51, 50)
    hist, edges = np.histogram(arr, bins=bins)
    for edge, h in zip(edges, hist):
        bar = "#" * (h * 50 // max(hist))  # scale到最大50字符宽
        print(f"{edge:4d}-{edge+50:4d} : {h:4d} {bar}")

if __name__ == "__main__":
    main()

In [ ]:
    print(f"Total sequences: {len(arr)}")
    print(f"Min length    : {arr.min()}")
    print(f"Max length    : {arr.max()}")
    print(f"Mean length   : {arr.mean():.2f}")
    print(f"Median length : {np.median(arr):.2f}")
    print(f"Q1 (25%)      : {np.percentile(arr, 25):.2f}")
    print(f"Q3 (75%)      : {np.percentile(arr, 75):.2f}")


In [ ]:
import numpy as np

# 加载
data = np.load("/Users/shulei/PycharmProjects/Dataset/scripts/test_set/pairwise_identity_out/similarity_matrix_upper.npz")

print("Keys in file:", data.files)

# 看第一个数组
M = data[data.files[0]]
print("Shape:", M.shape)
print("Dtype:", M.dtype)

# 如果是矩阵，可以直接看
if M.ndim == 2:
    print("Matrix example:\n", M[:5, :5])
elif M.ndim == 1:
    print("1D array length:", len(M))

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
从上三角 npz(i,j,v,n) 还原相似度矩阵，并导出“每条序列的最近邻相似度”。

功能：
- 读取 i, j, v, n
- 还原对称 N×N 矩阵（float32），diag=1.0
- 计算最近邻（忽略对角），导出 CSV
- 打印分位数与直方图计数
- 可选保存完整矩阵为 .npz

注意：
- 自动检测索引是否 0/1 基（若最大索引 == n，则视为 1 基，自动减一）
- N≈500 时用稠密矩阵完全没问题
"""

from pathlib import Path
import numpy as np
import pandas as pd

# ======== 用户参数（改这里）========
INPUT_UPPER_NPZ = "/Users/shulei/PycharmProjects/Dataset/scripts/test_set/pairwise_identity_out/similarity_matrix_upper.npz"
SAVE_NEIGHBOR_CSV = "/Users/shulei/PycharmProjects/Dataset/scripts/test_set/pairwise_identity_out/nearest_neighbor_from_upper.csv"
SAVE_MATRIX_NPZ = None   # 若不想保存整矩阵，设为 None
# 直方图区间（与之前风格一致）
BINS = np.array([0.0, 0.5, 0.7, 0.85, 0.9, 0.95, 0.98, 0.99, 0.995, 1.001], dtype=np.float32)
# =================================

def main():
    data = np.load(INPUT_UPPER_NPZ)
    required = {"i", "j", "v", "n"}
    if not required.issubset(set(data.files)):
        raise RuntimeError(f"npz keys={data.files} 缺少必需项 {required}")

    i = data["i"]
    j = data["j"]
    v = data["v"]
    n = int(np.array(data["n"]).item())  # 兼容标量数组

    if not (len(i) == len(j) == len(v)):
        raise ValueError("i/j/v 长度不一致")

    # 索引基准检测：若最大索引 == n，视为 1 基，需要减一
    max_idx = int(max(i.max(), j.max()))
    if max_idx == n:
        # 1-based -> 0-based
        i = i - 1
        j = j - 1
        max_idx = int(max(i.max(), j.max()))
    # 现在应满足 max_idx <= n-1
    if max_idx != n - 1:
        # 宽松检查：允许上三角不含最后一行/列的情况，但至少不应超界
        if max_idx >= n:
            raise ValueError(f"索引越界：max_idx={max_idx} >= n={n}")

    # 还原对称矩阵（float32）
    M = np.zeros((n, n), dtype=np.float32)
    # 上三角填入
    M[i, j] = v.astype(np.float32)
    # 镜像到下三角
    M[j, i] = v.astype(np.float32)
    # 对角置 1.0（自相似）
    np.fill_diagonal(M, 1.0)

    # 计算最近邻（忽略对角）
    diag_backup = M.diagonal().copy()
    np.fill_diagonal(M, -np.inf)
    nn_idx = M.argmax(axis=1)            # 每行最大列索引
    nn_sim = M[np.arange(n), nn_idx]     # 对应的最大相似度
    np.fill_diagonal(M, diag_backup)

    # 统计摘要
    q = np.quantile(nn_sim, [0, .25, .5, .75, .9, .95, .99, 1.0])
    print("[Nearest-Neighbor identity stats]")
    print(f"min={q[0]:.3f}  Q1={q[1]:.3f}  median={q[2]:.3f}  Q3={q[3]:.3f}  "
          f"P90={q[4]:.3f}  P95={q[5]:.3f}  P99={q[6]:.3f}  max={q[7]:.3f}")

    # 直方图计数（与之前风格对应）
    hist, edges = np.histogram(nn_sim, bins=BINS)
    for a, b, c in zip(edges[:-1], edges[1:], hist):
        left = "[" if a == edges[0] else "("
        right = "]" if b == edges[-1] else ")"
        print(f"\"{left}{a:.3f},{b:.3f}{right}\",{int(c)}")

    # 导出最近邻表
    df_nn = pd.DataFrame({
        "seq_index": np.arange(n, dtype=int),
        "nn_index": nn_idx.astype(int),
        "nn_identity": nn_sim.astype(np.float32)
    })
    Path(SAVE_NEIGHBOR_CSV).parent.mkdir(parents=True, exist_ok=True)
    df_nn.to_csv(SAVE_NEIGHBOR_CSV, index=False)
    print(f"[OK] 最近邻结果已保存: {SAVE_NEIGHBOR_CSV}")

    # 可选：保存整矩阵，便于后续分层抽样/可视化
    if SAVE_MATRIX_NPZ:
        Path(SAVE_MATRIX_NPZ).parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(SAVE_MATRIX_NPZ, identity=M)
        print(f"[OK] 完整相似度矩阵已保存: {SAVE_MATRIX_NPZ}")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
path = "/scripts/test_set/matrix_out/sparse_matrix_v0.3.1_observed_mask_csr.npz"  # 换成你CONFIG里的路径
with np.load(path) as z:
    print("keys:", z.files)

In [ ]:
# -*- coding: utf-8 -*-
"""
Compare test/train distribution at polymer-enzyme pair level
-----------------------------------------------------------
Inputs:
 - test_sequences.csv  (pipeline output, must have a 'sequence' column)
 - PlaszymeDB_v0.3.1.csv (original dataset with 'sequence' + 'plastic')

Outputs:
 - Console summary
 - distribution_summary_pairs.csv (plastic-wise pair counts)
"""

import pandas as pd

# === 路径配置 ===
TEST_CSV = "/Users/shulei/PycharmProjects/Dataset/scripts/test_set/run3/pipeline_out/test_sequences.csv"
ORIG_CSV = "/Users/shulei/PycharmProjects/Dataset/dataset/PlaszymeDB_v0.3.1._deduplicated.csv"
OUT_CSV = "./distribution_summary_pairs.csv"

# === 读取数据 ===
test_df = pd.read_csv(TEST_CSV)
orig_df = pd.read_csv(ORIG_CSV)

# 标准化列名
for df in (test_df, orig_df):
    df.columns = [c.strip().lower() for c in df.columns]

if "sequence" not in test_df.columns:
    raise ValueError("test_sequences.csv must contain 'sequence' column.")
if not {"sequence", "plastic"}.issubset(orig_df.columns):
    raise ValueError("Original CSV must contain 'sequence' and 'plastic' columns.")

# === 标记 test/train ===
test_set = set(test_df["sequence"].str.upper())
orig_df["is_test"] = orig_df["sequence"].str.upper().isin(test_set)
test_pairs = orig_df[orig_df["is_test"]]
train_pairs = orig_df[~orig_df["is_test"]]

# === 数量统计 ===
print(f"✅ 总对数: {len(orig_df)}")
print(f"🧪 测试集对数: {len(test_pairs)}")
print(f"📘 训练集对数: {len(train_pairs)}")

# === 塑料分布 ===
dist_test = test_pairs["plastic"].value_counts().rename("test_count")
dist_train = train_pairs["plastic"].value_counts().rename("train_count")
dist_total = orig_df["plastic"].value_counts().rename("total_count")

dist_df = pd.concat([dist_total, dist_train, dist_test], axis=1).fillna(0).astype(int)
dist_df["test_frac"] = dist_df["test_count"] / dist_df["total_count"]
dist_df["train_frac"] = dist_df["train_count"] / dist_df["total_count"]

print("\n=== 塑料分布 (pair 数量) ===")
print(dist_df)

# === 保存输出 ===
dist_df.to_csv(OUT_CSV)
print(f"\n📁 已保存分布表到 {OUT_CSV}")